In [2]:
from pathlib import Path
import pandas as pd
import pybedtools

In [25]:
ROOT = Path().cwd()
DATA = Path("../data").resolve()
DATA_INITIAL = Path("../data_initial").resolve()
BIOMART = DATA / "biomart"
BIOMART_INITIAL = DATA_INITIAL / "biomart"

Так как предыдущая версия Himorna была сделана на GRCh37, то и gene name для генов lncRNA также имеют прошлую натацию. Для обновления, воспользуемся biomart: \
https://grch37.ensembl.org/biomart/martview \
https://www.ensembl.org/biomart/martview

In [26]:
biomart_assemblies = {}
for i in ["GRCh37", "GRCh38"]:
    to_file = BIOMART_INITIAL / f"mart_export-{i}.tsv.gz"
    df = pd.read_csv(to_file, sep="\t")[["Gene stable ID", "Gene name"]].drop_duplicates()
    biomart_assemblies[i] = df

In [43]:
biomart_merged = pd.merge(left=biomart_assemblies["GRCh37"], right=biomart_assemblies["GRCh38"], 
         left_on="Gene stable ID", right_on="Gene stable ID", 
         how="outer")
biomart_merged.rename(columns={"Gene stable ID":"gene_ens",
                          "Gene name_x":"gene_name-GRCh37",
                          "Gene name_y":"gene_name-GRCh38"}, inplace=True)

In [45]:
biomart_merged

,gene_ens,gene_name-GRCh37,gene_name-GRCh38
0,ENSG00000000003,TSPAN6,TSPAN6
1,ENSG00000000005,TNMD,TNMD
2,ENSG00000000419,DPM1,DPM1
3,ENSG00000000457,SCYL3,SCYL3
4,ENSG00000000460,C1orf112,FIRRM
...,...,...,...
96422,ENSG00000310553,NaN,NaN
96423,ENSG00000310554,NaN,NaN
96424,ENSG00000310555,NaN,NaN
96425,ENSG00000310556,NaN,NaN


In [46]:
biomart_merged[biomart_merged["gene_ens"] == "ENSG00000234546"]

,gene_ens,gene_name-GRCh37,gene_name-GRCh38
35625,ENSG00000234546,RP3-510D11.2,LNCTAM34A


In [47]:
to_save = BIOMART / f"mart_export-GRCh37_to_GRCh38.tsv.gz"
biomart_merged.to_csv(to_save, sep="\t", compression="gzip", index=False)